# AI Job Market: Summary Statistics

This notebook summarizes job frequency, salary distribution, and remote-work patterns in the AI job market dataset.

In [1]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("../ai_jobs_market_2025_2026.csv")
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (1500, 25)


,job_id,job_title,job_category,experience_level,years_of_experience,education_required,annual_salary_usd,salary_min_usd,salary_max_usd,city,...,ai_salary_premium_pct,demand_score,demand_growth_yoy_pct,benefits_score_10,posting_year,posting_month,is_senior,is_remote_friendly,is_llm_role,salary_tier
0,AIJOB0001,AI Agent Developer,AI Engineering,Senior (6-9 yrs),7,Master's,239000.0,155000,290000,Boston,...,13.1,96,16.9,6.8,2026,3,1,0,1,Senior ($200-300k)
1,AIJOB0002,Prompt Engineer,AI Engineering,Senior (6-9 yrs),2,Bachelor's,166000.0,90000,200000,London,...,5.4,82,11.6,6.2,2026,1,1,1,1,Upper-Mid ($150-200k)
2,AIJOB0003,LLM Engineer,AI Engineering,Senior (6-9 yrs),4,Associate's,360000.0,160000,300000,Seattle,...,9.1,98,42.7,7.7,2026,1,1,1,1,Elite (>$300k)
3,AIJOB0004,Data Engineer (AI),Data Engineering,Senior (6-9 yrs),3,Bachelor's,161000.0,130000,220000,Singapore,...,12.0,88,6.7,9.5,2026,3,1,1,0,Upper-Mid ($150-200k)
4,AIJOB0005,AI Product Manager,Product,Lead (10+ yrs),5,Bootcamp/Self-taught,283000.0,140000,260000,Los Angeles,...,9.4,85,17.3,8.9,2026,1,1,1,0,Senior ($200-300k)


## 1. Overall salary distribution

We compare the mean and median salary to understand the typical salary level.

In [3]:
salary_summary = df["annual_salary_usd"].describe()
salary_summary

count      1500.000000
mean     194892.000000
std       66506.822013
min       90000.000000
25%      144750.000000
50%      180000.000000
75%      236250.000000
max      384000.000000
Name: annual_salary_usd, dtype: float64

In [4]:
mean_salary = df["annual_salary_usd"].mean()
median_salary = df["annual_salary_usd"].median()
difference = mean_salary - median_salary

print(f"Mean salary: ${mean_salary:,.0f}")
print(f"Median salary: ${median_salary:,.0f}")
print(f"Difference: ${difference:,.0f}")

Mean salary: $194,892
Median salary: $180,000
Difference: $14,892


### Observation

The mean annual salary is $194,892, while the median is $180,000.
The higher mean suggests that high-paying roles pull the average upward,
so the median better represents a typical job in this dataset.

In [5]:
job_counts = df["job_title"].value_counts()
job_counts.head(10)

job_title
LLM Engineer              75
Robotics Engineer (AI)    74
Prompt Engineer           71
Generative AI Engineer    71
AI Product Manager        70
Multimodal AI Engineer    67
Senior Data Scientist     66
AI Compliance Manager     66
Senior ML Engineer        64
AI Engineer               64
Name: count, dtype: int64

In [6]:
number_of_titles = df["job_title"].nunique()

top_job = job_counts.index[0]
top_count = job_counts.iloc[0]
top_share = top_count / len(df)

print("Number of unique job titles:", number_of_titles)
print("Most common job title:", top_job)
print("Number of postings:", top_count)
print(f"Share of all postings: {top_share:.1%}")

Number of unique job titles: 25
Most common job title: LLM Engineer
Number of postings: 75
Share of all postings: 5.0%


### Observation

The dataset contains 25 unique job titles. LLM Engineer is the most
common role with 75 postings, representing 5.0% of all records.
No single job title dominates the dataset.

## 3. Remote-work distribution

We compare the number and percentage of fully remote, hybrid, and on-site jobs.

In [7]:
remote_counts = df["remote_work"].value_counts()

remote_percentages = (
    df["remote_work"]
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
)

remote_summary = pd.DataFrame({
    "job_count": remote_counts,
    "percentage": remote_percentages
})

remote_summary

,job_count,percentage
remote_work,,
Hybrid,686,45.7
Fully Remote,445,29.7
On-site,369,24.6


In [8]:
print("Total jobs:", remote_summary["job_count"].sum())
print("Total percentage:", remote_summary["percentage"].sum())

Total jobs: 1500
Total percentage: 100.0


### Observation

Hybrid work is the most common arrangement, accounting for 686 jobs
or 45.7% of the dataset. Fully remote roles represent 29.7%, while
on-site roles account for 24.6%. Overall, 75.4% of the postings offer
either hybrid or fully remote work.

## 4. Salary position relative to the median

We label each job according to whether its salary is below, equal to,
or above the dataset median.

In [11]:
median_salary = df["annual_salary_usd"].median()

def classify_salary(salary):
    if salary > median_salary:
        return "Above median"
    elif salary < median_salary:
        return "Below median"
    else:
        return "Equal to median"

In [12]:
df["salary_position"] = df["annual_salary_usd"].map(classify_salary)

df[["annual_salary_usd", "salary_position"]].head(10)

,annual_salary_usd,salary_position
0,239000.0,Above median
1,166000.0,Below median
2,360000.0,Above median
3,161000.0,Below median
4,283000.0,Above median
5,155000.0,Below median
6,197000.0,Above median
7,221000.0,Above median
8,145000.0,Below median
9,90000.0,Below median


In [13]:
df["salary_position"].value_counts()

salary_position
Below median       734
Above median       728
Equal to median     38
Name: count, dtype: int64

### Observation

Using the median salary of $180,000 as the benchmark, 728 jobs are
above the median, 734 are below it, and 38 are exactly equal to it.
The nearly even split is expected because the median divides an
ordered dataset into two approximately equal halves.

## 5. Number of required skills

We extract the number of listed skills from each job posting and
summarize the resulting distribution.

In [3]:
df[["job_title", "required_skills"]].head()


,job_title,required_skills
0,AI Agent Developer,APIs|Planning Systems|Python|Cloud|SQL|Leadership
1,Prompt Engineer,Python|Documentation|LLM APIs|Prompt Design|NL...
2,LLM Engineer,Vector DBs|Python|Prompt Engineering|Fine-tuni...
3,Data Engineer (AI),Feature Stores|Spark|ETL|Airflow|dbt|SQL|Pytho...
4,AI Product Manager,Data Analysis|Stakeholder Mgmt|Agile|Cloud|Pro...


In [4]:
def count_skills(skills_text):
    skills = skills_text.split("|")
    return len(skills)

df["skill_count"] = df["required_skills"].apply(count_skills)

df[
    ["job_title", "required_skills", "skill_count"]
].head()

,job_title,required_skills,skill_count
0,AI Agent Developer,APIs|Planning Systems|Python|Cloud|SQL|Leadership,6
1,Prompt Engineer,Python|Documentation|LLM APIs|Prompt Design|NL...,7
2,LLM Engineer,Vector DBs|Python|Prompt Engineering|Fine-tuni...,5
3,Data Engineer (AI),Feature Stores|Spark|ETL|Airflow|dbt|SQL|Pytho...,10
4,AI Product Manager,Data Analysis|Stakeholder Mgmt|Agile|Cloud|Pro...,6


In [5]:
df["skill_count"].describe()

count    1500.000000
mean        6.365333
std         1.355393
min         4.000000
25%         5.000000
50%         6.000000
75%         7.000000
max        10.000000
Name: skill_count, dtype: float64

In [6]:
df["skill_count"].value_counts().sort_index()

skill_count
4     136
5     270
6     417
7     368
8     213
9      88
10      8
Name: count, dtype: int64

### Observation

AI job postings list an average of 6.37 required skills, with a
median of 6. Requirements range from 4 to 10 skills, and the most
common value is 6, appearing in 417 postings.